In [7]:
import json
import requests
from bs4 import BeautifulSoup
from sklearn.utils.class_weight import compute_class_weight
from transformers import BertTokenizer
from torch.utils.data import Dataset
import torch
import nltk
from tqdm import tqdm
from transformers import BertForSequenceClassification, AdamW, get_scheduler
from torch.utils.data import DataLoader
from torch.nn import CrossEntropyLoss
from torch.nn.utils import clip_grad_norm_
from torch.cuda.amp import GradScaler, autocast
import torch
from tqdm import tqdm
import numpy as np

In [2]:
nltk.download("punkt")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\hasan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [3]:
label_map = {
    "Refuted": 0,
    "Supported": 1,
    "Not Enough Evidence": 2,
    "Conflicting Evidence/Cherrypicking": 3
}

scrape_cache = {}
def scrape_url(entry):
    """Try cached_original_claim_url first, then original_claim_url."""
    for url_key in ["cached_original_claim_url", "original_claim_url"]:
        url = entry.get(url_key, "")
        if not url:
            continue
        if url in scrape_cache:
            return scrape_cache[url]
        try:
            headers = {"User-Agent": "Mozilla/5.0"}
            response = requests.get(url, headers=headers, timeout=5)
            soup = BeautifulSoup(response.text, "html.parser")
            texts = [p.get_text() for p in soup.find_all("p")]
            full_text = " ".join(texts)
            scrape_cache[url] = full_text
            return full_text
        except Exception:
            continue
    return ""

In [4]:
def format_qa(entry):
    """Format question-answer pairs into a single string."""
    qa_texts = []
    for q in entry.get("questions", []):
        question = q.get("question", "")
        for a in q.get("answers", []):
            answer = a.get("answer", "")
            qa_texts.append(f"Q: {question} A: {answer}")
    return " ".join(qa_texts)

with open("train.json", "r", encoding="utf-8") as f:
    train_data = json.load(f)

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
input_texts = []
input_labels = []

for entry in tqdm(train_data):
    claim = entry.get("claim", "")
    speaker = entry.get("speaker", "")
    date = entry.get("claim_date", "")
    ctype = ", ".join(entry.get("claim_types", []))
    source = entry.get("reporting_source", "")
    justification = entry.get("justification", "")
    strategies = ", ".join(entry.get("fact_checking_strategies", []))
    qa_text = format_qa(entry)
    label = label_map.get(entry["label"], -1)
    url_text = scrape_url(entry)

    combined = (
        f"Claim: {claim} Speaker: {speaker} Date: {date} "
        f"Type: {ctype} Source: {source} Justification: {justification} "
        f"FactCheckStrategy: {strategies} OriginalContent: {url_text} Evidence: {qa_text}"
    )

    input_texts.append(combined)
    input_labels.append(label)

tokenized = tokenizer(
    input_texts,
    padding=True,
    truncation=True,
    max_length=512,
    return_tensors="pt"
)

100%|██████████| 3068/3068 [1:01:17<00:00,  1.20s/it] 


In [ ]:
class ClaimDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = ClaimDataset(tokenized, input_labels)
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1, 2, 3]),
    y=np.array(input_labels)
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)

{
    "num_train_samples": len(train_dataset),
    "example_input_snippet": input_texts[0][:300] + "...",
    "class_weights": class_weights_tensor.tolist()
}

{'num_train_samples': 3068,
 'example_input_snippet': 'Claim: Hunter Biden had no experience in Ukraine or in the energy sector when he joined the board of Burisma. Speaker: Pam Bondi Date: 25-8-2020 Type: Position Statement Source: Speech at The Republican National Convention Justification: No former experience stated. FactCheckStrategy: Written Eviden...',
 'class_weights': [0.44029849767684937,
  0.9034157991409302,
  2.719858169555664,
  3.933333396911621]}

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [ ]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 4060 Laptop GPU


In [12]:
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=4)
model.to(device)
loss_fn = CrossEntropyLoss(weight=class_weights_tensor.to(device))
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
optimizer = AdamW(model.parameters(), lr=2e-5)
num_epochs = 3
num_training_steps = num_epochs * len(train_loader)
lr_scheduler = get_scheduler("linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)

scaler = GradScaler()
model.train()
for epoch in range(num_epochs):
    total_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()

        with autocast():
            outputs = model(**batch)
            loss = outputs.loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        lr_scheduler.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1} | Average Loss: {avg_loss:.4f}")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
c:\Users\hasan\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
C:\Users\hasan\AppData\Local\Temp\ipykernel_17488\3615862601.py:10: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
Epoch 1:   0%|          | 0/384 [00:00<?, ?it/s]C:\Users\hasan\AppData\Local\Temp\ipykernel_17488\3615862601.py:18: FutureWarning: `torch.cuda.amp

Epoch 1 | Average Loss: 0.7736


Epoch 2: 100%|██████████| 384/384 [01:53<00:00,  3.39it/s]


Epoch 2 | Average Loss: 0.3602


Epoch 3: 100%|██████████| 384/384 [01:48<00:00,  3.55it/s]

Epoch 3 | Average Loss: 0.2032


In [13]:
model.save_pretrained("averitec_bert_model")
tokenizer.save_pretrained("averitec_bert_model")

('averitec_bert_model\\tokenizer_config.json',
 'averitec_bert_model\\special_tokens_map.json',
 'averitec_bert_model\\vocab.txt',
 'averitec_bert_model\\added_tokens.json')

In [14]:
with open("dev.json", "r", encoding="utf-8") as f:
    dev_data = json.load(f)

In [16]:
input_texts = []
claim_ids = []

for entry in tqdm(dev_data):
    claim = entry.get("claim", "")
    speaker = entry.get("speaker", "")
    date = entry.get("claim_date", "")
    ctype = ", ".join(entry.get("claim_types", []))
    source = entry.get("reporting_source", "")
    justification = entry.get("justification", "")
    strategies = ", ".join(entry.get("fact_checking_strategies", []))
    qa_text = format_qa(entry)
    url_text = scrape_url(entry)

    combined = (
        f"Claim: {claim} Speaker: {speaker} Date: {date} "
        f"Type: {ctype} Source: {source} Justification: {justification} "
        f"FactCheckStrategy: {strategies} OriginalContent: {url_text} Evidence: {qa_text}"
    )

    input_texts.append(combined)
    claim_ids.append(entry.get("id", f"claim_{len(claim_ids)}"))


100%|██████████| 500/500 [09:44<00:00,  1.17s/it]


In [17]:
encoded = tokenizer(
    input_texts,
    padding=True,
    truncation=True,
    max_length=512,
    return_tensors="pt"
)

with torch.no_grad():
    outputs = model(**{k: v.to(device) for k, v in encoded.items()})
    preds = torch.argmax(outputs.logits, dim=1).cpu().tolist()

In [ ]:
label_map_inv = {0: "Refuted", 1: "Supported", 2: "Not Enough Evidence", 3: "Conflicting Evidence/Cherrypicking"}

final_output = []
for idx, entry in enumerate(dev_data):
    qa_list = []
    for q in entry.get("questions", []):
        question = q.get("question", "")
        for a in q.get("answers", []):
            answer = a.get("answer", "")
            url = a.get("source_url", "")
            qa_list.append({
                "question": question,
                "answer": answer,
                "url": url,
                "scraped_text": "" 
            })
    final_output.append({
        "id": claim_ids[idx],
        "verdict": label_map_inv[preds[idx]],
        "evidence": qa_list
    })


In [19]:
with open("dev_predictions_averitec.json", "w", encoding="utf-8") as f:
    json.dump(final_output, f, indent=2)

In [20]:
import json

with open("dev_predictions_averitec.json", "r", encoding="utf-8") as f:
    data = json.load(f)

for item in data:
    item["pred_label"] = item.pop("verdict")

with open("dev_predictions_averitec_fixed.json", "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2)


In [21]:
import json

# Load original predictions
with open("dev_predictions_averitec.json", "r", encoding="utf-8") as f:
    predictions = json.load(f)

# Convert "verdict" to "pred_label"
for item in predictions:
    if "verdict" in item:
        item["pred_label"] = item.pop("verdict")

# Save to a new file
with open("dev_predictions_fixed.json", "w", encoding="utf-8") as f:
    json.dump(predictions, f, indent=2)


In [23]:
with open("test.json", "r", encoding="utf-8") as f:
    test_data = json.load(f)

In [25]:
for entry in tqdm(test_data):
    claim = entry.get("claim", "")
    speaker = entry.get("speaker", "")
    date = entry.get("claim_date", "")
    ctype = ", ".join(entry.get("claim_types", []))
    source = entry.get("reporting_source", "")
    justification = entry.get("justification", "")
    strategies = ", ".join(entry.get("fact_checking_strategies", []))
    qa_text = format_qa(entry)
    url_text = scrape_url(entry)

    combined = (
        f"Claim: {claim} Speaker: {speaker} Date: {date} "
        f"Type: {ctype} Source: {source} Justification: {justification} "
        f"FactCheckStrategy: {strategies} OriginalContent: {url_text} Evidence: {qa_text}"
    )

    input_texts.append(combined)
    claim_ids.append(entry.get("id", f"claim_{len(claim_ids)}"))

100%|██████████| 2215/2215 [08:18<00:00,  4.44it/s]


In [ ]:
encoded = tokenizer(
    input_texts,
    padding=True,
    truncation=True,
    max_length=512,
    return_tensors="pt"
)

dataset = TensorDataset(encoded["input_ids"], encoded["attention_mask"])
loader = DataLoader(dataset, batch_size=8)  # batch_size safe on CPU

all_preds = []

model.eval()
with torch.no_grad():
    for batch in loader:
        input_ids, attention_mask = batch  # on CPU
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1).tolist()
        all_preds.extend(preds)

# ---------- Format AVeriTeC output ----------
test_output = []
for idx, entry in enumerate(test_data):
    qa_list = []
    for q in entry.get("questions", []):
        question = q.get("question", "")
        for a in q.get("answers", []):
            answer = a.get("answer", "")
            url = a.get("source_url", "")
            qa_list.append({
                "question": question,
                "answer": answer,
                "url": url,
                "scraped_text": ""  # optional
            })
    test_output.append({
        "id": claim_ids[idx],
        "pred_label": label_map_inv[all_preds[idx]],
        "evidence": qa_list
    })
with open("test_predictions_averitec.json", "w", encoding="utf-8") as f:
    json.dump(test_output, f, indent=2)